<a href="https://colab.research.google.com/github/AdLucem/AdLucem.github.io/blob/master/ire_miniproj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Download the data.

In [54]:
!rm -rf wikidata
!curl https://drive.google.com/file/d/1-jilaN3NPFsM7sF-5hyLmxhO9RYRMZAB/view?usp=sharing \
  --output wikidata_1.xml.bz2
!bunzip2 wikidata_1.xml.bz2

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70716    0 70716    0     0   124k      0 --:--:-- --:--:-- --:--:--  124k
bunzip2: wikidata_1.xml.bz2 is not a bzip2 file.


In [3]:
!pip install nltk
!pip install pystemmer

     |████████████████████████████████| 563kB 2.6MB/s 
  Created wheel for pystemmer: filename=PyStemmer-2.0.1-cp36-cp36m-linux_x86_64.whl size=422967 sha256=a6e077bfaaa77942dfa1993fc79aaf0577c4c84dd899c51753fbdbe7d17cb30e
  Stored in directory: /root/.cache/pip/wheels/f3/3c/11/ee323a09706e17a649c2730bd8819b06e887411ff7507acf7a
Successfully built pystemmer


## Indexer

Utility functions for performing the various adding-to-index and index-writing operations.

Imports

In [4]:
import re

from nltk.corpus import stopwords
import Stemmer

import nltk
nltk.download('stopwords')

"""XML schema for the Wiki Dump:

root:
    -> siteinfo
    -> [page]
"""

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


'XML schema for the Wiki Dump:\n\nroot:\n    -> siteinfo\n    -> [page]\n'

Initializations for indexer

In [5]:
sw = stopwords.words("english")
stemmer = Stemmer.Stemmer('english')
delims = "[\"\'-\;\!\?\/\{\}\[\]\(\)\\\.\,\:\#\s]"
nonalpha = "[\`\"'\-\;\!\?\/\{\}\[\]\(\)\\\.\,\:\#\s\@\~\$\%\^\&\*\_\+\=\<\>\|]"


Function to process and index a single token

In [6]:
def processToken(index, token, docID, fieldID):

    token = re.sub(nonalpha, "", token)
    token = token.replace('\u2014', '')

    # this takes out a lot of legit years and stuff
    # so refine this function
    token = re.sub("[0-9]", "", token)

    stem = stemmer.stemWord(token.lower())

    # adding to inverted index
    if stem in index:
            
        # if document ID in inverted index of word
        if docID in index[stem]:
            if fieldID in index[stem][docID]:
                index[stem][docID][fieldID] += 1
            else:
                index[stem][docID][fieldID] = 1
        # else add doc ID to index of word
        else:
            index[stem][docID] = {}
            index[stem][docID][fieldID] = 1

    # else add word to inverted index
    else:
        index[stem] = {docID: {}}
        index[stem][docID][fieldID] = 1

Process a single line as a list of tokens

In [7]:
def process(index, docID, fieldID, tokens):

    # remember that lists are 0-indexed
    fieldID -= 1

    for token in tokens[1:100]:

        if token in sw:
            pass

        else:
            processToken(index, token, docID, fieldID)

Process each article as a list of (newline-separated) lines

In [8]:
def processLines(index, docID, title, lines):

    """
    fieldID:
        1: Title
        2: Body
        3: Infobox
        4: Categories
        5: External links
        6: References
    """

    fieldID = 2
    for line in lines:
        
        if line == "== External links ==":
            fieldID = 5
        elif line == "== References ==" or \
                line == "== Citations ==" or \
                line == "== Bibliography ==":
            fieldID = 6
        elif re.match("[[Category:[a-zA-Z0-9 ]+]]", 
                      line):
            category = re.sub("\[\[Category:", "", line)
            category = re.sub("\]\]", "", category)
            tokens = re.split(delims, category)
            process(index, docID, 4, tokens)
        elif re.match("\{\{Infobox", line):
            box = re.sub("\{\{Infobox", "", line)
            box = re.sub("}}", "", box)
            tokens = re.split(delims, box)
            process(index, docID, 3, box)
        else:
            line = re.sub("[[}{]|]", "", line)
            tokens = re.split(delims, line)
            process(index, docID, fieldID, tokens)

Write a particular inverted index to a file

In [9]:
def writeIndex(index, write_to, write_terms_to):

    terms = list(index.keys())
    terms.sort()

    with open(write_to, "w+") as f:

        termID = 0
        for term in terms:
            f.write("$^$" + str(termID) + "\n")

            for docID in index[term]:
                f.write(str(docID) + "\n")

                freqs = ""
                
                for i in range(6):
                    if i in index[term][docID]:
                        freqs = freqs + \
                            str(index[term][docID][i]) + ","
                    else:
                        freqs = freqs + "0,"
                f.write(freqs + "\n")

            termID += 1

            f.write("----\n")

    with open(write_terms_to, "w+") as f:

        termID = 0

        for term in terms:
            f.write(str(termID) + " " + term + "\n")
            termID += 1


## Index A Full FIle

Index an entire file as `n` blocks of specified memory size

Imports (not including imports from `Indexer` file

In [10]:
import sys
from os.path import join

import xml.etree.ElementTree as ET

Index the inverted index for a single block

In [11]:
def write_block(index_dir, filenum, n_blocks, index):

    WRITE_INDEX_TO = join(index_dir,
                          str(filenum) + "_" + str(n_blocks)) + ".txt"
    WRITE_TERMS_TO = join(index_dir,
                          str(filenum) + "_" + str(n_blocks)) + "terms.txt"

    writeIndex(index, WRITE_INDEX_TO, WRITE_TERMS_TO)

Iterate over all the files in a dump, and run block-sort-based-indexing on them.

In [18]:
def iterate_and_index(max_blocks, increment, block_size, index_dir, filename, filenum):

    # initialize index
    index = {}

    n_blocks = 0
    # number of title/text pairs parsed
    n_ttl, n_txt = 0, 0
    # to get the root ID of page
    titleflag = 0

    # for the sake of input, make a list of docID-titles
    title_file = open(index_dir + "/titles" + str(filenum) + ".txt", "w+")

    for event, elem in ET.iterparse(filename):

        if 'title' in elem.tag:

            title = elem.text
            n_ttl += 1

            titleflag = 1

        elif ('id' in elem.tag) and titleflag:

            article_id = elem.text
            title_file.write(article_id + "|" + title + "\n")

            titleflag = 0

        elif "text" in elem.tag:

            try: 
                lines = elem.text.split("\n")
            except AttributeError:
                lines = []

            processLines(index, article_id, title, lines)
            n_txt += 1

        # after every <increment> iterations
        if n_txt % increment == 0:

            # get size of index
            idx_size = float(sys.getsizeof(index)) / 1048576.0
            # print(idx_size)

            # temp: break when 10 blocks written
            if n_blocks >= max_blocks:
                break
            if idx_size >= block_size:

                # write the current block
                write_block(index_dir, filenum, n_blocks, index)

                print("Written block " + str(n_blocks))

                # re-iniitialize index and increment block no
                index = {}
                n_blocks += 1
            #else:
            #    print("Not writing block at iteration", n_txt)

    # write the last block, after the whole file is read
    write_block(index_dir, filenum, n_blocks + 1, index)

## Run The Indexing Code

Run the indexing code on `n` files in a data directory

Imports

In [19]:
import sys
import time
import json
from os.path import join

Code to block-sort-index a single file and mark it (in a meta file) as sorted.

In [20]:
def index_one_file(data_dir, index_dir, stats_dir):

    # start measuring time
    t0 = time.clock()

    # name of file indexed in this iteration
    cur_indexed = ""

    # load meta information
    meta_file = join(index_dir, "meta.json")
    meta_info = json.load(open(meta_file))
    cur_file_no = int(meta_info[0])
    meta = meta_info[1]
    print(cur_file_no, meta)

    for file in meta:
        if meta[file] == 0.0:

            cur_indexed = file
            iterate_and_index(10,
                              # float('inf'),  # max. blocks
                              1000,  # increment after which to check block size
                              1.0,  # size of each block in MB
                              index_dir,
                              file,
                              cur_file_no)  # data file
            break

    # finish measuring time
    t1 = time.clock()

    # if a file has been indexed in this iteration
    if cur_indexed != "":
        # store meta information about file
        meta[cur_indexed] = t1 - t0
        json.dump([cur_file_no + 1, meta], open(meta_file, "w+"))
        print("Time taken: ", t1 - t0)
        return 0
    else:
        print("Indexing all done")
        return 1

## Initialize The Inverted Index

Create an inverted index directory and initialise a meta file containing information on all files to be indexed.

In [21]:
!rm -rf inverted_index
!mkdir wikidata
!mv wikidata_1.xml wikidata/

import json
import sys
from os import mkdir, listdir
from os.path import isfile, join


def setup(index_dir, data_dir):
    """Make an index directory and a meta file"""

    mkdir(index_dir)

    meta_file = join(index_dir, "meta.json")

    # get all unzipped files in data directory
    datafiles = [join(data_dir, f) for
                 f in listdir(data_dir)
                 if isfile(join(data_dir, f))]

    # mark all files as unindexed
    meta = {}
    for file in datafiles:
        meta[file] = 0

    json.dump([0, meta], open(meta_file, "w+"))


!rm -rf inverted_index
DATA_DIR = "wikidata"
INDEX_DIR = "inverted_index"
setup(INDEX_DIR, DATA_DIR)

mkdir: cannot create directory ‘wikidata’: File exists
mv: cannot stat 'wikidata_1.xml': No such file or directory


## Run Inverted Indexing

Run the inverted index code on all the files in a directory. Run this code as many times as necessary to index all the files in a data directory.

In [22]:
DATA_DIR = "wikidata"
INDEX_DIR = "inverted_index"
WRITE_STATS_TO = "inverted_index/stats.txt"

done = index_one_file(DATA_DIR, INDEX_DIR, WRITE_STATS_TO)

if not done:
    print("More files to index. Run `index.sh` again.")


0 {'wikidata/wikidata_1.xml': 0}
Written block 0
Written block 1
Written block 2
Written block 3
Written block 4
Written block 5
Written block 6
Written block 7
Written block 8
Written block 9
Time taken:  621.268897
More files to index. Run `index.sh` again.


In [ ]:
def has_elements(tags):

    has_title = False
    has_id = False
    has_text = False

    for tag in tags:

        if 'title' in tag:
            has_title = True
        if 'id' in tag:
            has_id = True
        if 'text' in tag:
            has_text = True

    return (has_title and has_id and has_text)


def iterate_and_index(max_blocks, increment, block_size, index_dir, filename, filenum):

    # initialize index
    index = {}

    n_blocks = 0
    # number of title/text pairs parsed
    n_ttl, n_txt = 0, 0
    # to get the root ID of page
    titleflag = 0

    # for the sake of input, make a list of docID-titles
    title_file = open(index_dir + "/titles" + str(filenum) + ".txt", "w+")
    i = 0
    for event, elem in ET.iterparse(filename):

        if 'page' in elem.tag:
            i += 1
            subels = list(elem.iter())
            tags = list(map(lambda x: x.tag, subels))
            # print(tags)

            if not has_elements(tags):
                print("ERROR!")

            if i % 10000 == 0:
              print(i)

            elem.clear()

        if i == 1000000:
            break

       #  elem.clear()


iterate_and_index(10,
                  # float('inf'),  # max. blocks
                  10,  # increment after which to check block size
                  0.1,  # size of each block in MB
                  "inverted_index",
                  "wikidata/wikidata_1.xml",
                  0)  # data file

In [31]:
import random
import string
import re
import os
from os import listdir
from os.path import isfile, join

# make (file, word) pairs
# take top `n` keys that are the same
# get corresponding file pointers
# merge all values from those file pointers
# write to opened merged-index file
# for only files in #topvals, 
# move dict pointers to next word

In [32]:
def read_posting(f):
    """Reading posting in a way that points out
    errors in the indexing"""

    # first line will be termID
    try:
        term_line = f.readline()
        termID = int(term_line.replace("$^$", ""))
    except ValueError:
        return -1, {}
    
    # next we loop through docIDs and fieldFreqs
    # until we find a '----' delimiter
    docID = f.readline()
    posting = ""

    while docID[:-1] != '----':

        try:
            docID_ = int(docID[:-1])  # remove newline
            fields_str = f.readline()
            fields = fields_str.split(",")
            freqs = [int(n) for n in fields 
                     if ((n != "") and (n != "\n"))]
            if freqs == []:
                raise ValueError
            posting = posting + docID + fields_str
            docID = f.readline()
        except ValueError:
            print("Something went wrong in reading posting")
            break

    return termID, posting

In [33]:
def merge_postings(values):
    """this can be done much faster by keeping
    the postings as strings"""

    m_posting = ""

    for value in values:
        m_posting += "|".join(value.split("\n"))

    m_posting += "\n"

    return m_posting

In [34]:
def merge(dict_ptrs, idx_ptrs, 
          merged_ptr, merged_idx_ptr, 
          o_term_id, word, topfiles):
    # print(o_term_id, word)
    to_merge, idx_ptrs = get_postings(topfiles, idx_ptrs)

    for file in topfiles:
        dict_ptrs[file].readline()

    merged = merge_postings(to_merge)
    merged_ptr.write(str(o_term_id) + "|" + merged)
    merged_idx_ptr.write(str(o_term_id) + " " + word + "\n")
    o_term_id += 1
    
    return (dict_ptrs, idx_ptrs, merged_ptr, 
            merged_idx_ptr, o_term_id)

In [35]:
def get_postings(files, idx_ptrs):
    """Get next term-postingsList for
    all file in files"""

    postings = []
    for file in files:
        termID, posting = read_posting(idx_ptrs[file])
        postings.append(posting)

    return postings, idx_ptrs

In [ ]:
def get_top_keys(pairs):
    """Get keys for all value = top_value
    in dict, where dict is represented
    as (key, value) pairs.
    remember value is a (termID, term) pair"""

    topval = pairs[0][1][1]

    # print("Chosen val: ", topval)
    # print("-----------------------------")

    topkeys = []
    for key, value in pairs:
        if value[1] == topval:
            topkeys.append(key)
        else:
            break

    return topval, topkeys

In [36]:
def get_current_values(pointers):
    """Get current values to be processed
    for all files in pointers"""

    pairs = []
    for file in pointers:

        prev = pointers[file].tell()
        line = pointers[file].readline()

        if line != '':

            termID, word = line.rstrip("\n").split(" ")
            pointers[file].seek(prev)
            pairs.append((file, (termID, word)))

    pairs.sort(key=lambda x: x[1][1])

    return pairs

In [43]:
def get_dict_pointers(index_dir):
    """Get pointers to all term dictionary
    files in the index directory"""

    files = [join(index_dir, f) for 
             f in listdir(index_dir) 
             if isfile(join(index_dir, f))]
    dicts = [f for 
             f in files 
             if re.search("[0-9]\_[0-9]terms\.txt", f)]
    print(dicts)

    pointers = {}
    for d in dicts:
        f = open(d)
        d_no = int(d.split("/")[1].rstrip("terms.txt"))
        pointers[d_no] = f

    return pointers

In [45]:
def get_index_pointers(index_dir):
    """Get pointers to all index
    files in the index directory"""

    files = [join(index_dir, f) for 
             f in listdir(index_dir) 
             if isfile(join(index_dir, f))]
    indexes = [f for 
               f in files 
               if re.search("[0-9]\_[0-9]\.txt", f)]
    print(indexes)

    pointers = {}
    for index in indexes:
        f = open(index)
        index_no = int(index.split("/")[1]
                       .rstrip(".txt"))
        pointers[index_no] = f

    return pointers

In [40]:
def print_pointers(pointers):
    for file in pointers:
        print(pointers[file].tell(), end=" ")
    print()


def print_terms(words):

    for word in words:
        print(word[1][1], end=" | ")
    print()

In [47]:
def main(dirname):
    # pointers to dict files
    pointers = get_dict_pointers(dirname)
    # index pointers
    idx_ptrs = get_index_pointers(dirname)
    # open file for merged index
    merged_ptr = open(dirname + "/merged.txt", "w+")
    merged_idx_ptr = open(dirname + "/merged_index.txt", "w+")
    # pointer for the current overall termID
    overall_term_id = 0

    while True:
      for i in range(100):
        words = get_current_values(pointers)

        print_terms(words)
"""
        if words == []:
            print("done")
            break

        word, topfiles = get_top_keys(words)
                
        (pointers, 
         idx_ptrs, 
         merged_ptr, 
         merged_idx_ptr,
         overall_term_id) = merge(pointers,
                                  idx_ptrs,
                                  merged_ptr,
                                  merged_idx_ptr,
                                  overall_term_id,
                                  word,
                                  topfiles)

        # print_pointers(pointers)

    with open(dirname + "/stats.txt", "w+") as f:
        f.write(str(overall_term_id))

    # close all the files
    for file in pointers:
        pointers[file].close()
    for file in idx_ptrs:
        idx_ptrs[file].close()
    merged_ptr.close()
    merged_idx_ptr.close()
"""

'\n        if words == []:\n            print("done")\n            break\n\n        word, topfiles = get_top_keys(words)\n                \n        (pointers, \n         idx_ptrs, \n         merged_ptr, \n         merged_idx_ptr,\n         overall_term_id) = merge(pointers,\n                                  idx_ptrs,\n                                  merged_ptr,\n                                  merged_idx_ptr,\n                                  overall_term_id,\n                                  word,\n                                  topfiles)\n\n        # print_pointers(pointers)\n\n    with open(dirname + "/stats.txt", "w+") as f:\n        f.write(str(overall_term_id))\n\n    # close all the files\n    for file in pointers:\n        pointers[file].close()\n    for file in idx_ptrs:\n        idx_ptrs[file].close()\n    merged_ptr.close()\n    merged_idx_ptr.close()\n'

In [53]:
!head -n 1000 wikidata/wikidata_1.xml
#main("inverted_index")

<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.10/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.mediawiki.org/xml/export-0.10/ http://www.mediawiki.org/xml/export-0.10.xsd" version="0.10" xml:lang="en">
  <siteinfo>
    <sitename>Wikidata</sitename>
    <dbname>wikidatawiki</dbname>
    <base>https://www.wikidata.org/wiki/Wikidata:Main_Page</base>
    <generator>MediaWiki 1.35.0-wmf.37</generator>
    <case>first-letter</case>
    <namespaces>
      <namespace key="-2" case="first-letter">Media</namespace>
      <namespace key="-1" case="first-letter">Special</namespace>
      <namespace key="0" case="first-letter" />
      <namespace key="1" case="first-letter">Talk</namespace>
      <namespace key="2" case="first-letter">User</namespace>
      <namespace key="3" case="first-letter">User talk</namespace>
      <namespace key="4" case="first-letter">Wikidata</namespace>
      <namespace key="5" case="first-letter">Wikidata talk</namespa